In [1]:
import os
os.chdir(r"D:\doc\2026\FINAL_CODE\src")
print(os.getcwd())

D:\doc\2026\FINAL_CODE\src


In [ ]:
import os
import re
import glob
import math
import time
import numpy as np
import pandas as pd
from scipy.io import wavfile as _wavfile
import librosa
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
import codeStat.modelimports.modelFunction as func
from codeStat.modelimports.modelFunction import *

In [3]:
dir_save_path="D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/nonfix_out/GMM"

In [4]:
# Load CSV
df_csv = pd.read_csv("D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv")

# Buat dictionary
dict_transkrip = {
    os.path.splitext(os.path.basename(path))[0]: kalimat
    for path, kalimat in zip(df_csv["Path_Data"], df_csv["sentences"])
}

def baca_transkrip(wav_path: str) -> str:
    audio_id = os.path.splitext(os.path.basename(wav_path))[0]
    return dict_transkrip.get(audio_id, "").lower()

In [ ]:
SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\GMM_TDNN\out\GMM"
)

In [ ]:
SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

CSV_PATH = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\preModelling"
    r"\samplingStratified\out_samplingStratified\dataSampledSmall.csv"
)

DIR_SAVE = (
    r"D:\doc\2026\FINAL_CODE\src\codeStat\modelling"
    r"\voiceModelling\srcVoiceModelling\GMM_TDNN\out\GMM"
)

N_COMPONENTS     = 16     # K komponen GMM per kata
N_ITER           = 100    # maks iterasi EM
TOL              = 1e-4   # toleransi konvergensi
N_GRAM           = 3      # orde N-gram (trigram)
TOP_K_DECODE     = 10     # kandidat kata untuk N-gram rescore
JALANKAN_TUNING  = True   # jalankan hyperparameter tuning (bisa dimatikan)


# ══════════════════════════════════════════════════════════════════════════════
# IMPORT STANDAR
# ══════════════════════════════════════════════════════════════════════════════

import os
import re
import glob
import math
import time
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

try:
    import librosa
    _LIBROSA = True
except ImportError:
    _LIBROSA = False
    print("[WARNING] librosa tidak ditemukan. Install: pip install librosa")

try:
    from scipy.io import wavfile as _wavfile
    _SCIPY = True
except ImportError:
    _SCIPY = False


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 1 — NORMALISASI & LEXICON
# Sub-bab 3.3.6 skripsi
# ══════════════════════════════════════════════════════════════════════════════

PETA_FONEM: Dict[str, List[str]] = {
    "ng": ["ng"], "ny": ["ny"], "th": ["th"], "dh": ["dh"],
    "a": ["a"],  "b": ["b"],  "c": ["ch"], "d": ["d"],
    "e": ["e"],  "f": ["f"],  "g": ["g"],  "h": ["h"],
    "i": ["i"],  "j": ["j"],  "k": ["k"],  "l": ["l"],
    "m": ["m"],  "n": ["n"],  "o": ["o"],  "p": ["p"],
    "r": ["r"],  "s": ["s"],  "t": ["t"],  "u": ["u"],
    "v": ["v"],  "w": ["w"],  "y": ["y"],  "z": ["z"],
}


def normalisasi(teks: str) -> str:
    """Lowercase, hapus non-huruf, normalisasi spasi."""
    teks = teks.lower().strip()
    teks = re.sub(r"[^a-z\s]", "", teks)
    return re.sub(r"\s+", " ", teks).strip()


def kata_ke_fonem(kata: str) -> List[str]:
    """Kata → urutan fonem. Digraf (ng, ny, th, dh) diprioritaskan."""
    fonem, i = [], 0
    while i < len(kata):
        if i + 1 < len(kata) and kata[i: i + 2] in PETA_FONEM:
            fonem.extend(PETA_FONEM[kata[i: i + 2]])
            i += 2
        elif kata[i] in PETA_FONEM:
            fonem.extend(PETA_FONEM[kata[i]])
            i += 1
        else:
            i += 1
    return fonem if fonem else [kata]


def buat_lexicon(transkrip_list: List[str]) -> Dict[str, List[str]]:
    """Buat kamus {kata: [fonem]} dari semua transkrip."""
    lexicon: Dict[str, List[str]] = {}
    for kalimat in transkrip_list:
        for kata in normalisasi(kalimat).split():
            if kata not in lexicon:
                lexicon[kata] = kata_ke_fonem(kata)
    return lexicon


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 2 — EKSTRAKSI MFCC
# Sub-bab 3.3.4 skripsi — 39 dimensi (13 MFCC + 13 delta + 13 delta-delta)
# ══════════════════════════════════════════════════════════════════════════════

_SR         = 16000
_N_MFCC     = 13
_N_FFT      = 512
_HOP_LENGTH = 160
_WIN_LENGTH = 400
_N_MELS     = 26
_PRE_EMPH   = 0.97
_FMIN       = 0.0
_FMAX       = 4000.0
_DELTA_W    = 2


def extract_features(wav_path: str) -> np.ndarray:
    """
    Ekstrak MFCC 39-dim dari file WAV/FLAC.
    Menggunakan librosa jika tersedia, fallback ke scipy+numpy.

    Returns:
        np.ndarray shape (T, 39)
    """
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")

    if _LIBROSA:
        return _mfcc_librosa(wav_path)
    elif _SCIPY:
        return _mfcc_manual(wav_path)
    else:
        raise ImportError("Install librosa atau scipy: pip install librosa scipy")


def _mfcc_librosa(wav_path: str) -> np.ndarray:
    """Ekstrak MFCC dengan librosa (lebih cepat dan stabil)."""
    y, _ = librosa.load(wav_path, sr=_SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio kosong: {wav_path}")

    # Pre-emphasis: X(n) = y(n) - 0.97 × y(n-1)  [formula 3.3.4.1.1]
    y = np.append(y[0], y[1:] - _PRE_EMPH * y[:-1])

    mfcc    = librosa.feature.mfcc(
        y=y, sr=_SR, n_mfcc=_N_MFCC,
        n_fft=_N_FFT, hop_length=_HOP_LENGTH,
        win_length=_WIN_LENGTH, n_mels=_N_MELS,
        fmin=_FMIN, fmax=_FMAX, window="hamming",
    )
    delta   = librosa.feature.delta(mfcc, width=_DELTA_W * 2 + 1)
    delta2  = librosa.feature.delta(mfcc, order=2, width=_DELTA_W * 2 + 1)

    return np.vstack([mfcc, delta, delta2]).T.astype(np.float32)


def _mfcc_manual(wav_path: str) -> np.ndarray:
    """Fallback manual MFCC tanpa librosa."""
    sr, audio = _wavfile.read(wav_path)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # stereo → mono

    if sr != _SR:
        n_new = int(len(audio) * _SR / sr)
        audio = np.interp(
            np.linspace(0, len(audio) - 1, n_new),
            np.arange(len(audio)), audio
        )

    y = np.append(audio[0], audio[1:] - _PRE_EMPH * audio[:-1])

    # Framing
    n_frames = max(1, 1 + (len(y) - _WIN_LENGTH) // _HOP_LENGTH)
    frames   = np.zeros((n_frames, _WIN_LENGTH), dtype=np.float32)
    for i in range(n_frames):
        s = i * _HOP_LENGTH
        e = s + _WIN_LENGTH
        chunk = y[s:e]
        frames[i, :len(chunk)] = chunk

    frames *= np.hamming(_WIN_LENGTH).astype(np.float32)

    fft_mag = np.abs(np.fft.rfft(frames, n=_N_FFT))  # (T, N_FFT//2+1)

    # Mel filter bank
    def hz2mel(f): return 1127.0 * np.log1p(f / 700.0)
    def mel2hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts = np.linspace(hz2mel(_FMIN), hz2mel(_FMAX), _N_MELS + 2)
    hz_pts  = mel2hz(mel_pts)
    bins    = np.floor((_N_FFT + 1) * hz_pts / _SR).astype(int)
    fb      = np.zeros((_N_MELS, _N_FFT // 2 + 1), dtype=np.float32)
    for m in range(1, _N_MELS + 1):
        for k in range(bins[m - 1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / max(bins[m] - bins[m-1], 1)
        for k in range(bins[m], bins[m + 1]):
            fb[m-1, k] = (bins[m+1] - k) / max(bins[m+1] - bins[m], 1)

    mel_e = np.maximum(fft_mag ** 2 @ fb.T, 1e-10)
    log_e = np.log10(mel_e)  # (T, N_MELS)

    # DCT → MFCC
    N = _N_MELS
    dct_m = np.cos(np.pi * np.arange(_N_MFCC)[:, None]
                   * (np.arange(N) + 0.5)[None, :] / N)
    mfcc  = (log_e @ dct_m.T) * np.sqrt(2.0 / N)  # (T, N_MFCC)

    # Delta
    def delta_feat(f, w=_DELTA_W):
        T, D  = f.shape
        denom = 2.0 * sum(n**2 for n in range(1, w + 1))
        d     = np.zeros_like(f)
        for t in range(T):
            for n in range(1, w + 1):
                d[t] += n * (f[min(t+n, T-1)] - f[max(t-n, 0)])
        return d / denom

    d1 = delta_feat(mfcc)
    d2 = delta_feat(d1)
    return np.concatenate([mfcc, d1, d2], axis=1).astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 3 — GAUSSIAN MIXTURE MODEL
# Sub-bab 3.3.5.1 skripsi — EM Algorithm, formula 3.3.5.1.2 s/d 3.3.5.1.9
# ══════════════════════════════════════════════════════════════════════════════

class GaussianMixtureModel:
    """
    GMM dengan diagonal covariance, implementasi manual (tanpa sklearn).

    Parameters
    ----------
    n_components : K komponen Gaussian
    n_iter       : maks iterasi EM
    tol          : toleransi konvergensi log-likelihood
    reg_covar    : regularisasi covariance
    random_state : seed random
    """

    def __init__(
        self,
        n_components: int = 4,
        n_iter      : int = 100,
        tol         : float = 1e-3,
        reg_covar   : float = 1e-6,
        random_state: Optional[int] = None,
    ):
        self.n_components = n_components
        self.n_iter       = n_iter
        self.tol          = tol
        self.reg_covar    = reg_covar
        self.random_state = random_state

        self.weights_    : Optional[np.ndarray] = None   # (K,)
        self.means_      : Optional[np.ndarray] = None   # (K, D)
        self.covariances_: Optional[np.ndarray] = None   # (K, D) diagonal
        self.converged_  : bool  = False
        self.n_iter_done_: int   = 0
        self.log_likelihood_history_: list = []

    def _init_params(self, X: np.ndarray):
        rng = np.random.default_rng(self.random_state)
        T, D = X.shape
        K    = self.n_components
        idx  = rng.choice(T, size=K, replace=False)
        self.means_       = X[idx].copy()
        self.weights_     = np.full(K, 1.0 / K)
        self.covariances_ = np.tile(np.var(X, axis=0) + self.reg_covar, (K, 1))

    def _log_gaussian(self, X: np.ndarray, k: int) -> np.ndarray:
        """log N(x; μ_k, Σ_k) untuk semua frame — diagonal covariance."""
        D   = X.shape[1]
        mu  = self.means_[k]
        var = self.covariances_[k]
        log_det  = np.sum(np.log(var))
        diff     = X - mu
        maha     = np.sum(diff ** 2 / var, axis=1)
        return -0.5 * (D * np.log(2 * np.pi) + log_det + maha)

    def _e_step(self, X: np.ndarray):
        T, K = X.shape[0], self.n_components
        log_w = np.zeros((T, K))
        for k in range(K):
            log_w[:, k] = np.log(self.weights_[k] + 1e-300) + self._log_gaussian(X, k)
        # log-sum-exp trick
        lse      = np.max(log_w, axis=1, keepdims=True)
        log_sum  = lse.squeeze() + np.log(np.sum(np.exp(log_w - lse), axis=1))
        log_resp = log_w - log_sum[:, None]
        return log_resp, log_sum

    def _m_step(self, X: np.ndarray, log_resp: np.ndarray):
        T, D = X.shape
        K    = self.n_components
        resp = np.exp(log_resp)
        Nk   = resp.sum(axis=0) + 1e-300

        self.weights_     = Nk / T
        self.means_       = (resp.T @ X) / Nk[:, None]

        new_cov = np.zeros((K, D))
        for k in range(K):
            diff       = X - self.means_[k]
            new_cov[k] = (resp[:, k, None] * diff ** 2).sum(axis=0) / Nk[k]
        self.covariances_ = new_cov + self.reg_covar

    def fit(self, X: np.ndarray) -> "GaussianMixtureModel":
        """Latih GMM dengan EM Algorithm."""
        assert X.ndim == 2, "X harus shape (T, D)"
        self.n_components = min(self.n_components, X.shape[0])
        self._init_params(X)
        self.log_likelihood_history_ = []
        prev_ll = -np.inf

        for it in range(self.n_iter):
            log_resp, log_sum = self._e_step(X)
            ll = float(log_sum.mean())
            self.log_likelihood_history_.append(ll)

            if abs(ll - prev_ll) < self.tol:
                self.converged_    = True
                self.n_iter_done_  = it + 1
                break
            prev_ll = ll
            self._m_step(X, log_resp)
        else:
            self.converged_    = False
            self.n_iter_done_  = self.n_iter

        return self

    def score(self, X: np.ndarray) -> float:
        """Rata-rata log p(X) — digunakan sebagai emission probability."""
        assert X.ndim == 2, "X harus shape (T, D)"
        K = self.n_components
        T = X.shape[0]
        log_w = np.zeros((T, K))
        for k in range(K):
            log_w[:, k] = np.log(self.weights_[k] + 1e-300) + self._log_gaussian(X, k)
        lse     = np.max(log_w, axis=1, keepdims=True)
        log_sum = lse.squeeze() + np.log(np.sum(np.exp(log_w - lse), axis=1))
        return float(log_sum.mean())


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 4 — N-GRAM LANGUAGE MODEL
# Sub-bab 3.3.7 skripsi — Kneser-Ney Smoothing (formula 2.20)
# ══════════════════════════════════════════════════════════════════════════════

class NGramModel:
    """Trigram Language Model dengan Kneser-Ney smoothing."""

    def __init__(self, n: int = 3, discount: float = 0.75):
        self.n        = n
        self.discount = discount
        self.ngram_count  : Dict[tuple, int] = defaultdict(int)
        self.context_count: Dict[tuple, int] = defaultdict(int)
        self.vocab        : set              = set()
        self.N_total      : int              = 0
        self._ctx_diversity: Dict[tuple, int] = {}

    def _tokenize(self, kalimat: str) -> List[str]:
        tokens = normalisasi(kalimat).split()
        return ["<s>"] * (self.n - 1) + tokens + ["</s>"]

    def fit(self, korpus: List[str]) -> "NGramModel":
        """Latih N-gram dari list kalimat."""
        ctx_words: Dict[tuple, set] = defaultdict(set)
        for kalimat in korpus:
            tokens = self._tokenize(kalimat)
            self.vocab.update(tokens)
            self.N_total += len(tokens)
            for i in range(len(tokens) - self.n + 1):
                ngram   = tuple(tokens[i: i + self.n])
                ctx     = ngram[:-1]
                self.ngram_count[ngram] += 1
                self.context_count[ctx] += 1
                ctx_words[ctx].add(ngram[-1])
        self._ctx_diversity = {c: len(w) for c, w in ctx_words.items()}
        return self

    def log_prob(self, kata: str, context: tuple) -> float:
        """log P(kata | context) dengan Kneser-Ney smoothing."""
        ngram   = context + (kata,)
        C_ng    = self.ngram_count.get(ngram, 0)
        C_ctx   = self.context_count.get(context, 0)

        if C_ctx == 0:
            C_uni = self.ngram_count.get((kata,), 0)
            return math.log((C_uni + 1) / (self.N_total + len(self.vocab)) + 1e-300)

        lam    = self.discount * self._ctx_diversity.get(context, 0) / C_ctx
        C_uni  = self.ngram_count.get((kata,), 0)
        p_back = (C_uni + 1) / (self.N_total + len(self.vocab))
        prob   = max(C_ng - self.discount, 0) / C_ctx + lam * p_back
        return math.log(prob + 1e-300)

    def score_sequence(self, kata_list: List[str]) -> float:
        """Total log-probability urutan kata."""
        tokens = ["<s>"] * (self.n - 1) + kata_list + ["</s>"]
        log_p  = 0.0
        for i in range(self.n - 1, len(tokens)):
            context = tuple(tokens[i - self.n + 1: i])
            log_p  += self.log_prob(tokens[i], context)
        return log_p

    def rescore(self, kandidat: List[List[str]]) -> List[str]:
        """Pilih urutan kata terbaik dari kandidat berdasarkan skor N-gram."""
        if not kandidat:
            return []
        return max(kandidat, key=self.score_sequence)


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 5 — WORD ERROR RATE
# Formula 3.3.9.1 skripsi: WER = (S + D + I) / N
# ══════════════════════════════════════════════════════════════════════════════

def compute_wer(referensi: List[str], hipotesis: List[str]) -> Dict:
    """
    Hitung WER dengan dynamic programming (Levenshtein distance).

    Returns dict: {wer_persen, S, D, I, N}
    """
    R, H = len(referensi), len(hipotesis)

    dp = [[0] * (H + 1) for _ in range(R + 1)]
    for i in range(R + 1): dp[i][0] = i
    for j in range(H + 1): dp[0][j] = j

    for i in range(1, R + 1):
        for j in range(1, H + 1):
            if referensi[i-1] == hipotesis[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])

    S = D = I = 0
    i, j = R, H
    while i > 0 or j > 0:
        if i > 0 and j > 0 and referensi[i-1] == hipotesis[j-1]:
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1

    N   = R
    wer = (S + D + I) / N if N > 0 else 0.0
    return {"wer_persen": round(wer * 100, 2), "S": S, "D": D, "I": I, "N": N}


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 6 — CACHE MFCC (OPTIMASI UTAMA)
# ══════════════════════════════════════════════════════════════════════════════

def baca_transkrip_csv(csv_path: str) -> Dict[str, str]:
    """Baca CSV → {audio_id: kalimat}."""
    df = pd.read_csv(csv_path)
    result = {}
    for _, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row["Path_Data"])))[0]
        result[audio_id] = str(row["sentences"]).lower().strip()
    return result


def build_mfcc_cache(
    sampled_root   : str,
    dict_transkrip : Dict[str, str],
) -> Tuple[Dict[str, np.ndarray], List[Dict]]:
    """
    Extract MFCC semua audio SEKALI → simpan di dict.
    Semua tahap selanjutnya baca dari dict ini, tidak buka file lagi.

    Returns:
        cache_mfcc : {audio_id: (T, 39)}
        meta_list  : list info untuk HasilMFCC.csv
    """
    wav_files  = glob.glob(os.path.join(sampled_root, "**/*.wav"),  recursive=True)
    wav_files += glob.glob(os.path.join(sampled_root, "**/*.flac"), recursive=True)

    cache_mfcc: Dict[str, np.ndarray] = {}
    meta_list : List[Dict]            = []
    n_total   = len(wav_files)

    print(f"  Mengekstrak MFCC dari {n_total} file ...")
    t0 = time.time()

    for idx, wav_path in enumerate(wav_files, 1):
        try:
            audio_id = os.path.splitext(os.path.basename(wav_path))[0]
            label    = dict_transkrip.get(audio_id, "")
            if not label:
                continue

            fitur = extract_features(wav_path)  # (T, 39)
            cache_mfcc[audio_id] = fitur
            meta_list.append({
                "Id"           : audio_id,
                "Label"        : label,
                "Jumlah Frame" : fitur.shape[0],
                "Dimensi Fitur": fitur.shape[1],
            })

            if idx % 100 == 0:
                ela = time.time() - t0
                eta = ela / idx * (n_total - idx)
                print(f"    [{idx}/{n_total}] cache={len(cache_mfcc)}  "
                      f"elapsed={ela:.0f}s  ETA={eta:.0f}s")

        except Exception as e:
            print(f"    [SKIP] {os.path.basename(wav_path)}: {e}")

    print(f"  Selesai: {len(cache_mfcc)} audio dalam {time.time()-t0:.1f}s")
    return cache_mfcc, meta_list


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 7 — FRAME PER KATA & LATIH GMM PER KATA
# ══════════════════════════════════════════════════════════════════════════════

def pecah_frame_per_kata(
    fitur    : np.ndarray,
    kata_list: List[str],
) -> Dict[str, np.ndarray]:
    """Bagi frame MFCC secara proporsional ke setiap kata."""
    T, n = fitur.shape[0], len(kata_list)
    if n == 0 or T == 0:
        return {}

    bobot = np.array([max(len(kata_ke_fonem(k)), 1) for k in kata_list], dtype=float)
    bobot /= bobot.sum()
    n_frame = (bobot * T).astype(int)
    sisa    = T - n_frame.sum()
    for i in range(sisa):
        n_frame[i % n] += 1

    hasil: Dict[str, np.ndarray] = {}
    idx = 0
    for i, kata in enumerate(kata_list):
        nf  = n_frame[i]
        seg = fitur[idx: idx + nf]
        if nf > 0:
            hasil[kata] = np.vstack([hasil[kata], seg]) if kata in hasil else seg
        idx += nf
    return hasil


def kumpulkan_frame_per_kata(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
) -> Dict[str, List[np.ndarray]]:
    """Kumpulkan frame per kata dari cache (tidak baca file)."""
    frame_per_kata: Dict[str, List[np.ndarray]] = defaultdict(list)

    print(f"  Mengumpulkan frame per kata dari {len(cache_mfcc)} audio ...")
    for audio_id, fitur in cache_mfcc.items():
        label = dict_transkrip.get(audio_id, "")
        if not label:
            continue
        kata_list = normalisasi(label).split()
        for kata, frame in pecah_frame_per_kata(fitur, kata_list).items():
            frame_per_kata[kata].append(frame)

    print(f"  Kata unik dengan frame: {len(frame_per_kata)}")
    return frame_per_kata


def latih_gmm_per_kata(
    frame_per_kata: Dict[str, List[np.ndarray]],
    n_components  : int   = 4,
    n_iter        : int   = 100,
    tol           : float = 1e-3,
    min_frame     : int   = 10,
    verbose       : bool  = True,
) -> Dict[str, GaussianMixtureModel]:
    """Latih satu GMM per kata."""
    gmm_per_kata: Dict[str, GaussianMixtureModel] = {}
    kata_list    = sorted(frame_per_kata.keys())
    n_kata       = len(kata_list)

    if verbose:
        print(f"  Melatih GMM untuk {n_kata} kata (K={n_components}) ...")
    t0 = time.time()

    for i, kata in enumerate(kata_list):
        X = np.vstack(frame_per_kata[kata])
        if X.shape[0] < min_frame:
            continue

        gmm = GaussianMixtureModel(
            n_components = min(n_components, X.shape[0]),
            n_iter       = n_iter,
            tol          = tol,
            reg_covar    = 1e-6,
            random_state = hash(kata) % 100_000,
        )
        gmm.fit(X)
        gmm_per_kata[kata] = gmm

        if verbose and (i + 1) % 50 == 0:
            print(f"    [{i+1}/{n_kata}] elapsed={time.time()-t0:.1f}s")

    if verbose:
        print(f"  GMM terlatih: {len(gmm_per_kata)} kata dalam {time.time()-t0:.1f}s")
    return gmm_per_kata


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 8 — DEKODE & WER
# ══════════════════════════════════════════════════════════════════════════════

def dekode_audio(
    fitur        : np.ndarray,
    kata_ref_list: List[str],
    gmm_per_kata : Dict[str, GaussianMixtureModel],
    ngram_model  : NGramModel,
) -> List[str]:
    """
    Dekode satu audio → urutan kata hipotesis.
    Referensi digunakan untuk segmentasi saja, BUKAN sebagai kandidat.
    """
    if not gmm_per_kata or not kata_ref_list:
        return []

    pecahan   = pecah_frame_per_kata(fitur, kata_ref_list)
    vocab_gmm = list(gmm_per_kata.keys())
    hasil     : List[str] = []

    for kata_seg in kata_ref_list:
        seg = pecahan.get(kata_seg, None)
        if seg is None or seg.shape[0] == 0:
            hasil.append(vocab_gmm[0] if vocab_gmm else kata_seg)
            continue

        # Hitung skor semua GMM kata → pilih terbaik
        skor    = {k: gmm_per_kata[k].score(seg) for k in vocab_gmm}
        best    = max(skor, key=skor.get)
        hasil.append(best)

    # N-gram rescore — HANYA hipotesis GMM, bukan referensi
    return ngram_model.rescore([hasil])


def dekode_semua_audio(
    cache_mfcc    : Dict[str, np.ndarray],
    dict_transkrip: Dict[str, str],
    gmm_per_kata  : Dict[str, GaussianMixtureModel],
    ngram_model   : NGramModel,
    verbose       : bool = True,
) -> List[Dict]:
    """Dekode semua audio dari cache + hitung WER."""
    hasil_wer = []
    n_total   = len(cache_mfcc)
    t0        = time.time()

    if verbose:
        print(f"  Dekode {n_total} audio ...")

    for idx, (audio_id, fitur) in enumerate(cache_mfcc.items(), 1):
        try:
            label = dict_transkrip.get(audio_id, "")
            if not label:
                continue
            ref_kata = normalisasi(label).split()
            if not ref_kata:
                continue

            hyp_kata  = dekode_audio(fitur, ref_kata, gmm_per_kata, ngram_model)
            wer_hasil = compute_wer(ref_kata, hyp_kata)

            hasil_wer.append({
                "Id"       : audio_id,
                "Referensi": " ".join(ref_kata),
                "Hipotesis": " ".join(hyp_kata),
                "WER (%)"  : wer_hasil["wer_persen"],
                "S"        : wer_hasil["S"],
                "D"        : wer_hasil["D"],
                "I"        : wer_hasil["I"],
                "N"        : wer_hasil["N"],
            })

            if verbose and idx % 100 == 0:
                print(f"    [{idx}/{n_total}] elapsed={time.time()-t0:.1f}s")

        except Exception as e:
            if verbose:
                print(f"    [ERROR] {audio_id}: {e}")

    if verbose:
        print(f"  Selesai: {len(hasil_wer)} audio dalam {time.time()-t0:.1f}s")
    return hasil_wer


def ringkasan_wer(df_wer: pd.DataFrame, label: str = "") -> Dict:
    """Cetak dan kembalikan statistik WER."""
    if df_wer.empty:
        print("  [WARNING] Tidak ada data WER")
        return {}

    total_S    = int(df_wer["S"].sum())
    total_D    = int(df_wer["D"].sum())
    total_I    = int(df_wer["I"].sum())
    total_N    = int(df_wer["N"].sum())
    wer_corpus = (total_S + total_D + total_I) / total_N * 100 if total_N > 0 else 100.0

    print(f"\n── Ringkasan WER {label} ──────────────────────")
    print(f"  Jumlah audio  : {len(df_wer)}")
    print(f"  WER Corpus    : {wer_corpus:.2f}%")
    print(f"  WER Mean      : {df_wer['WER (%)'].mean():.2f}%")
    print(f"  WER Median    : {df_wer['WER (%)'].median():.2f}%")
    print(f"  WER Std       : {df_wer['WER (%)'].std():.2f}%")
    print(f"  WER Min       : {df_wer['WER (%)'].min():.2f}%")
    print(f"  WER Max       : {df_wer['WER (%)'].max():.2f}%")
    print(f"  Total S/D/I/N : {total_S}/{total_D}/{total_I}/{total_N}")
    print(f"{'─'*46}")

    return {
        "n_audio"   : len(df_wer),
        "wer_corpus": round(wer_corpus, 2),
        "wer_mean"  : round(df_wer["WER (%)"].mean(), 2),
        "wer_median": round(df_wer["WER (%)"].median(), 2),
        "wer_std"   : round(df_wer["WER (%)"].std(), 2),
        "wer_min"   : round(df_wer["WER (%)"].min(), 2),
        "wer_max"   : round(df_wer["WER (%)"].max(), 2),
        "total_S"   : total_S, "total_D": total_D,
        "total_I"   : total_I, "total_N": total_N,
    }


# ══════════════════════════════════════════════════════════════════════════════
# BAGIAN 9 — HYPERPARAMETER TUNING (EFISIEN)
# frame_per_kata & cache_mfcc sudah ada → hanya re-train GMM + re-decode
# ══════════════════════════════════════════════════════════════════════════════

def hyperparameter_tuning(
    frame_per_kata : Dict[str, List[np.ndarray]],
    cache_mfcc     : Dict[str, np.ndarray],
    dict_transkrip : Dict[str, str],
    ngram_model    : NGramModel,
    dir_save       : str = ".",
) -> pd.DataFrame:
    """
    Grid search K, tol, max_iter — sesuai Tabel 3.4.1 skripsi.
    MFCC tidak di-extract ulang. frame_per_kata tidak dikumpulkan ulang.
    """
    grid = [
        (K, tol, mi)
        for K  in [4, 8, 16, 32]
        for tol in [1e-3, 1e-4]
        for mi in [50, 100, 200]
    ]
    n_eksp = len(grid)
    print(f"\n  Grid: {n_eksp} eksperimen (tanpa re-extract MFCC)")

    hasil_tuning = []
    t0_total     = time.time()

    for idx, (K, tol, mi) in enumerate(grid, 1):
        t0 = time.time()
        print(f"\n  [{idx}/{n_eksp}] K={K}, tol={tol:.0e}, max_iter={mi}")

        # Hanya re-train GMM
        gmm_pk = latih_gmm_per_kata(
            frame_per_kata, n_components=K, n_iter=mi, tol=tol, verbose=False
        )

        # Hanya re-decode dari cache
        hw = dekode_semua_audio(
            cache_mfcc, dict_transkrip, gmm_pk, ngram_model, verbose=False
        )

        if hw:
            df_tmp     = pd.DataFrame(hw)
            errs       = df_tmp["S"].sum() + df_tmp["D"].sum() + df_tmp["I"].sum()
            total_N    = df_tmp["N"].sum()
            wer_corpus = errs / total_N * 100 if total_N > 0 else 100.0
        else:
            wer_corpus = 100.0

        ela = time.time() - t0
        print(f"         WER={wer_corpus:.2f}%  ({ela:.1f}s)")

        hasil_tuning.append({
            "Eksperimen"    : idx,
            "K Komponen"    : K,
            "Toleransi"     : tol,
            "Maks Iterasi"  : mi,
            "WER Corpus (%)": round(wer_corpus, 2),
            "Waktu (s)"     : round(ela, 1),
        })

    df_tuning = pd.DataFrame(hasil_tuning).sort_values("WER Corpus (%)")
    df_tuning.to_csv(f"{dir_save}/HasilHyperparameterTuning_GMM.csv", index=False)

    best = df_tuning.iloc[0]
    print(f"\n  ✓ Parameter Terbaik:")
    print(f"    K={int(best['K Komponen'])}, "
          f"tol={best['Toleransi']:.0e}, "
          f"iter={int(best['Maks Iterasi'])}, "
          f"WER={best['WER Corpus (%)']:.2f}%")
    print(f"  Total waktu tuning: {(time.time()-t0_total)/60:.1f} menit")
    return df_tuning


# ══════════════════════════════════════════════════════════════════════════════
# MAIN — Entry Point
# ══════════════════════════════════════════════════════════════════════════════

def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    t_start = time.time()

    print("=" * 60)
    print("  GMM ASR BAHASA JAWA — Pipeline Efisien (All-in-One)")
    print("=" * 60)

    # [1] Load transkrip
    print("\n[1] Membaca transkrip CSV ...")
    dict_transkrip  = baca_transkrip_csv(CSV_PATH)
    semua_transkrip = list(dict_transkrip.values())
    print(f"    Total audio: {len(dict_transkrip)}")

    lexicon = buat_lexicon(semua_transkrip)
    pd.DataFrame([
        {"Kata": k, "Fonem": " ".join(v)} for k, v in lexicon.items()
    ]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)
    print(f"    Vocab: {len(lexicon)} kata unik  →  HasilLexicon.csv ✓")

    # [2] Extract MFCC → cache (SEKALI SAJA)
    print("\n[2] Ekstraksi MFCC ke cache ...")
    cache_mfcc, meta_list = build_mfcc_cache(SAMPLED_ROOT, dict_transkrip)
    pd.DataFrame(meta_list).to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    print(f"    HasilMFCC.csv ✓  ({len(cache_mfcc)} audio di-cache)")

    if not cache_mfcc:
        print("  [ERROR] Tidak ada audio berhasil di-load. Cek SAMPLED_ROOT.")
        return

    # [3] Kumpulkan frame per kata (dari cache)
    print("\n[3] Mengumpulkan frame per kata ...")
    frame_per_kata = kumpulkan_frame_per_kata(cache_mfcc, dict_transkrip)

    # [4] GMM Global (untuk referensi)
    print("\n[4] Melatih GMM Global ...")
    X_global   = np.vstack(list(cache_mfcc.values()))
    print(f"    Shape X_global: {X_global.shape}")
    gmm_global = GaussianMixtureModel(
        n_components=N_COMPONENTS, n_iter=N_ITER, tol=TOL,
        reg_covar=1e-6, random_state=42
    )
    gmm_global.fit(X_global)
    pd.DataFrame([{
        "K Komponen"          : N_COMPONENTS,
        "Iterasi EM"          : gmm_global.n_iter_done_,
        "Konvergen"           : gmm_global.converged_,
        "Log-Likelihood Akhir": round(gmm_global.log_likelihood_history_[-1], 6),
    }]).to_csv(f"{DIR_SAVE}/HasilGMMGlobal.csv", index=False)
    print(f"    Konvergen={gmm_global.converged_}  "
          f"Iter={gmm_global.n_iter_done_}  "
          f"LL={gmm_global.log_likelihood_history_[-1]:.4f}  "
          f"→ HasilGMMGlobal.csv ✓")

    # [5] GMM per kata
    print(f"\n[5] Melatih GMM per kata (K={N_COMPONENTS}) ...")
    gmm_per_kata = latih_gmm_per_kata(
        frame_per_kata, n_components=N_COMPONENTS,
        n_iter=N_ITER, tol=TOL
    )
    hasil_gmm_pk = []
    for kata, gmm in gmm_per_kata.items():
        hasil_gmm_pk.append({
            "Kata"                : kata,
            "Jumlah Frame"        : np.vstack(frame_per_kata[kata]).shape[0],
            "K Komponen"          : gmm.n_components,
            "Iterasi EM"          : gmm.n_iter_done_,
            "Konvergen"           : gmm.converged_,
            "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 4),
        })
    pd.DataFrame(hasil_gmm_pk).to_csv(f"{DIR_SAVE}/HasilGMMPerKata.csv", index=False)
    print(f"    HasilGMMPerKata.csv ✓  ({len(hasil_gmm_pk)} kata)")

    # [6] N-Gram Language Model
    print(f"\n[6] Melatih {N_GRAM}-Gram Language Model ...")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    ngram_model.fit(semua_transkrip)
    print(f"    Vocab={len(ngram_model.vocab)}  N-gram={len(ngram_model.ngram_count)}")

    # [7] Dekode + WER
    print("\n[7] Dekode audio + hitung WER ...")
    hasil_wer = dekode_semua_audio(cache_mfcc, dict_transkrip, gmm_per_kata, ngram_model)
    df_wer    = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_GMM.csv", index=False)
    print(f"    HasilWER_GMM.csv ✓  ({len(df_wer)} audio)")

    stat = ringkasan_wer(df_wer, label="GMM per Kata")
    pd.DataFrame([stat]).to_csv(f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False)

    # [8] Hyperparameter Tuning
    if JALANKAN_TUNING:
        print("\n[8] Hyperparameter Tuning ...")
        hyperparameter_tuning(
            frame_per_kata, cache_mfcc, dict_transkrip,
            ngram_model, DIR_SAVE
        )
    else:
        print("\n[8] Tuning dilewati (JALANKAN_TUNING=False)")

    # Selesai
    total = time.time() - t_start
    print("\n" + "=" * 60)
    print(f"  SELESAI dalam {total/60:.1f} menit")
    print(f"  Output: {DIR_SAVE}")
    print("=" * 60)
    for f in ["HasilLexicon", "HasilMFCC", "HasilGMMGlobal",
              "HasilGMMPerKata", "HasilWER_GMM", "HasilRingkasanWER",
              "HasilHyperparameterTuning_GMM"]:
        path = f"{DIR_SAVE}/{f}.csv"
        status = "✓" if os.path.isfile(path) else "–"
        print(f"  {status}  {f}.csv")


if __name__ == "__main__":
    main()